## Init

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import to_date, avg, max, min, sum, round as spark_round

## Read from silver table

In [0]:
df_silver_read = spark.read.table("weather.silver.weather_hourly")

## Aggregation 1: Daily summary by city

In [0]:
df_gold_daily = (
    df_silver_read
    .groupBy("date", "state_name", "city_name")
    .agg(
        spark_round(avg("temperature_celsius"), 2).alias("avg_temperature"),
        spark_round(max("temperature_celsius"), 2).alias("max_temperature"),
        spark_round(min("temperature_celsius"), 2).alias("min_temperature"),
        spark_round(avg("humidity_percentage"), 2).alias("avg_humidity"),
        spark_round(sum("precipitation_mm"), 2).alias("total_precipitation_mm")
    )
)

## Write in gold table with merge (UPSERT)

In [0]:
# A. Initialize the gold table in Delta Lake if it is the first time it is being executed 
(
    df_gold_daily.write
        .mode("ignore")
        .format("delta")
        .saveAsTable("weather.gold.daily_city_summary")
)

In [0]:
# B. Execute MERGE using (city_name + date) as unique key 
target_table = DeltaTable.forName(spark, "weather.gold.daily_city_summary")

target_table.alias("target").merge(
    df_gold_daily.alias("source"),
    """
    target.city_name = source.city_name AND
    target.date = source.date 
    """
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()